## RAG Concept

What is RAG?

RAG = Retrieval Augmented Generation
= Finding relevant information + Using LLM to generate answer from that information

┌─────────────────────────────────────────┐
│              RAG PIPELINE               │
│                                         │
│  YOUR DOCUMENTS (PDF, CSV, Web, etc)    │
│         ↓                               │
│  INDEXING PIPELINE (one time)           │
│  Load → Split → Embed → Store          │
│         ↓                               │
│  VECTOR DATABASE                        │
│  (stores all document embeddings)       │
│         ↑                               │
│  QUERYING PIPELINE (every time)         │
│  Question → Embed → Search → Retrieve  │
│         ↓                               │
│  GENERATION                             │
│  Question + Retrieved Docs → LLM       │
│         ↓                               │
│  FINAL ANSWER ✅                        │
└─────────────────────────────────────────┘

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os


In [6]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

llm.invoke("Hello, how are you?").content

"I'm functioning properly, thank you for asking. I'm a large language model, so I don't have emotions or feelings like humans do, but I'm here to help answer any questions or provide information you might need. How can I assist you today?"

In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Python Programming.pdf")
docs = loader.load()
print(docs[10].page_content)
# "Rishi Kumar, B.Tech AI & DS..."

3.5.3 Run Python Scripts from the Command Prompt in Win-
dows . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 34
3.5.4 Run Python Scripts from Spyder . . . . . . . . . . . . . . 34
4 Basic Python Programming 37
4.1 Basic Python Program . . . . . . . . . . . . . . . . . . . . . . . . 37
4.1.1 Get Help . . . . . . . . . . . . . . . . . . . . . . . . . . . 37
4.2 Variables . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 37
4.2.1 Numbers . . . . . . . . . . . . . . . . . . . . . . . . . . . 39
4.2.2 Strings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 40
4.2.3 String Input . . . . . . . . . . . . . . . . . . . . . . . . . . 41
4.3 Built-in Functions . . . . . . . . . . . . . . . . . . . . . . . . . . 41
4.4 Python Standard Library . . . . . . . . . . . . . . . . . . . . . . 42
4.5 Using Python Libraries, Packages and Modules . . . . . . . . . . 43
4.5.1 Python Packages . . . . . . . . . . . . . . . . . . . . . . . 45
4.6 Plotting in Python . . . . . .

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # 1000 chars per chunk
    chunk_overlap=200     # 200 chars overlap
)

chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")

print(chunks)

Total chunks: 211
[Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-06-12T10:51:46+00:00', 'moddate': '2026-06-12T10:51:46+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'source': 'Python Programming.pdf', 'total_pages': 143, 'page': 0, 'page_label': '1'}, page_content='Python ProgrammingHans-Petter Halvorsen\nhttps://www.halvorsen.blog'), Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-06-12T10:51:46+00:00', 'moddate': '2026-06-12T10:51:46+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'source': 'Python Programming.pdf', 'total_pages': 143, 'page': 1, 'page_label': '2'}, page_content='Python Programming'), Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-06-12T10:51:46+00:00', 'moddate':

In [9]:
print("🔢 Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="thenlper/gte-small",
    model_kwargs={"device": "cpu"}
)
print("✅ Embedding model loaded!")

🔢 Loading embedding model...


C:\Users\rishi\AppData\Local\Temp\ipykernel_3512\800572617.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6439.98it/s]


✅ Embedding model loaded!


In [10]:
import os
import chromadb
from dotenv import load_dotenv

client = chromadb.CloudClient(
    api_key=os.getenv("CHROMA_API_KEY"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
)

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    client=client,
    embedding_function=embeddings,
    collection_name="python"
)

vectorstore.add_documents(chunks)

print(f"Stored {len(chunks)} chunks in Chroma Cloud!")

In [ ]:
results = vectorstore.similarity_search("What is recursion?", k=3)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful Python tutor.

Use only the provided context to answer the question.
If the answer is not found in the context, reply:
"Not found in the book."

Context:
{context}
"""
    ),
    (
        "user",
        "{question}"
    ),
])



from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()





In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {
        "context":  retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
answer = rag_chain.invoke("Who published the book?")
print(answer)

The book "Python Programming" was published by Hans-Petter Halvorsen.


In [ ]:
question = "Who published the book?"

print("LLM only:")
print(llm.invoke(question).content)

print("\nRAG:")
print(rag_chain.invoke(question))

LLM only:
I don't have any information about a book you're referring to. Could you please provide more context or details about the book you're asking about?

RAG:
The book "Python Programming" was published by Hans-Petter Halvorsen.


In [ ]:
query = "Who is the author of the book?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"Chunk {i}")
    print(doc.page_content)
    print("-" * 80)

Chunk 1
Python Programming
©Hans-Petter Halvorsen
June 12, 2026
ISBN:978-82-691106-4-7
139
--------------------------------------------------------------------------------
Chunk 2
Bibliography
[1] H.-P. Halvorsen, “Technology blog - https://www.halvorsen.blog,” 2018.
[2] H.-P. Halvorsen, “Technology blog - https://en.wikipedia.org/wiki/Python(programming language), ′′ 2018.
[3] T. . T. P. Languages, “The 2018 top programming languages
- https://spectrum.ieee.org/at-work/innovation/the-2018-top-
programming-languages,” 2018.
[4] S. Overflow, “Stack overflow developer survey 2018 -
https://insights.stackoverflow.com/survey/2018/,” 2018.
[5] stackoverflow.blog, “The incredible growth of python -
https://stackoverflow.blog/2017/09/06/incredible-growth-python/,”
2018.
[6] python.org, “python.org - https://www.python.org,” 2018.
[7] python.org, “The python tutorial - https://docs.python.org/3.7/tutorial/,”
2018.
[8] python.org, “Python 3.7.1 documentation - https://docs.python.org/3.7/,”
201